# Camada Silver — Tratamento e Padronização

## Objetivo

Transformar os registros brutos de interrupções de carga do ONS, armazenados na camada Bronze, em um conjunto de dados estruturado, consistente e adequado à modelagem dimensional.

A camada Silver preserva a granularidade dos registros originais, aplica regras de qualidade e disponibiliza dados tratados para a construção das tabelas fato e dimensão da camada Gold.

## Transformações previstas

- Conversão dos campos numéricos e temporais.
- Padronização de identificadores e atributos categóricos.
- Identificação e remoção de registros integralmente duplicados.
- Validação de medidas físicas e indicadores.
- Criação de atributos temporais auxiliares.
- Segregação dos registros inválidos para auditoria.

Os registros originais permanecem preservados na camada Bronze. Os registros rejeitados não serão apagados, mas armazenados separadamente para rastreabilidade.

In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "ons_energia"

TABELA_ORIGEM = f"{CATALOG}.bronze.bronze_interrupcao_carga"

TABELA_SILVER = f"{CATALOG}.silver.silver_interrupcao_carga"

TABELA_INVALIDOS = f"{CATALOG}.silver.silver_registros_invalidos"

df_bronze = spark.table(TABELA_ORIGEM)

print("Registros recebidos:", df_bronze.count())

display(df_bronze.limit(10))


## Conversão dos tipos de dados

Os atributos da camada Bronze foram preservados como texto para evitar interpretações automáticas durante a ingestão.

Nesta etapa, os campos são convertidos para tipos adequados às análises:

- Data e hora: `timestamp`.
- Potência interrompida: `double`.
- Tempo médio de recomposição: `double`.
- Energia não suprida: `double`.
- Identificadores e atributos categóricos: `string`.

Os campos numéricos utilizam conversão tolerante a erros. Valores incompatíveis serão convertidos em nulos e identificados posteriormente pelas regras de qualidade.

A conversão será realizada antes de qualquer agregação estatística.

In [0]:

def converter_numero(coluna):
    valor = F.trim(F.col(coluna))

    # Remove espaços e separadores de milhar, quando houver.
    valor = F.regexp_replace(valor, r"\s+", "")

    # Formato brasileiro: 1.234,56 -> 1234.56
    valor = F.when(
        F.instr(valor, ",") > 0,
        F.regexp_replace(
            F.regexp_replace(valor, r"\.", ""),
            ",",
            "."
        )
    ).otherwise(valor)

    return F.expr(
        f"try_cast({valor._jc.toString()} AS DOUBLE)"
    )



In [0]:

def numero_br(coluna):
    c = F.trim(F.col(coluna))
    c = F.regexp_replace(c, r"\s+", "")

    c = F.when(
        F.instr(c, ",") > 0,
        F.regexp_replace(
            F.regexp_replace(c, r"\.", ""),
            ",",
            "."
        )
    ).otherwise(c)

    return F.when(
        c.rlike(r"^[+-]?\d+(\.\d+)?$"),
        c.cast("double")
    ).otherwise(F.lit(None).cast("double"))


df_tipado = (
    df_bronze
    .withColumn(
        "cod_perturbacao",
        F.trim(F.col("cod_perturbacao"))
    )
    .withColumn(
        "id_subsistema",
        F.upper(F.trim(F.col("id_subsistema")))
    )
    .withColumn(
        "nom_subsistema",
        F.trim(F.col("nom_subsistema"))
    )
    .withColumn(
        "id_estado",
        F.upper(F.trim(F.col("id_estado")))
    )
    .withColumn(
        "nom_agente",
        F.trim(F.col("nom_agente"))
    )
    .withColumn(
        "flg_envolveuredebasica",
        F.upper(F.trim(F.col("flg_envolveuredebasica")))
    )
    .withColumn(
        "flg_envolveuredeoperacao",
        F.upper(F.trim(F.col("flg_envolveuredeoperacao")))
    )
    .withColumn(
        "din_interrupcaocarga",
        F.coalesce(
            F.try_to_timestamp(
                F.col("din_interrupcaocarga"),
                F.lit("dd/MM/yyyy HH:mm:ss")
            ),
            F.try_to_timestamp(
                F.col("din_interrupcaocarga"),
                F.lit("yyyy-MM-dd HH:mm:ss")
            )
        )
    )
    .withColumn(
        "val_cargainterrompida_mw",
        numero_br("val_cargainterrompida_mw")
    )
    .withColumn(
        "val_tempomedio_minutos",
        numero_br("val_tempomedio_minutos")
    )
    .withColumn(
        "val_energianaosuprida_mwh",
        numero_br("val_energianaosuprida_mwh")
    )
)

display(df_tipado.limit(10))

df_tipado.printSchema()


## Tratamento de duplicatas

A base pode conter múltiplos registros associados à mesma perturbação, uma vez que diferentes agentes, localidades ou subsistemas podem estar envolvidos.

Por esse motivo, o código da perturbação não será utilizado isoladamente como critério de unicidade.

Serão considerados duplicados os registros que apresentarem igualdade em todos os 11 atributos originais, após a padronização dos respectivos valores.

Os registros excedentes serão identificados e separados, preservando uma ocorrência de cada combinação de atributos.

In [0]:

colunas_negocio = [
    "cod_perturbacao",
    "din_interrupcaocarga",
    "id_subsistema",
    "nom_subsistema",
    "id_estado",
    "nom_agente",
    "val_cargainterrompida_mw",
    "val_tempomedio_minutos",
    "val_energianaosuprida_mwh",
    "flg_envolveuredebasica",
    "flg_envolveuredeoperacao"
]

qtd_antes = df_tipado.count()

df_sem_duplicatas = df_tipado.dropDuplicates(
    colunas_negocio
)

qtd_depois = df_sem_duplicatas.count()

print("Registros antes:", qtd_antes)
print("Registros após deduplicação:", qtd_depois)
print("Duplicatas removidas:", qtd_antes - qtd_depois)


## Regras de qualidade dos dados

Os registros serão avaliados conforme critérios de completude, consistência e plausibilidade física.

### Regras aplicadas

1. O identificador da perturbação não pode estar vazio.
2. A data da interrupção deve ser válida.
3. O subsistema deve estar identificado.
4. A UF deve possuir uma sigla válida.
5. A carga interrompida não pode ser negativa.
6. O tempo médio de recomposição não pode ser negativo.
7. A energia não suprida não pode ser negativa.
8. Os indicadores de envolvimento de rede devem conter `S` ou `N`.

Valores negativos nas medidas serão classificados como inválidos, pois não representam magnitudes físicas aceitáveis para os indicadores analisados.

Registros que não atenderem às regras serão segregados em uma tabela de auditoria. A Bronze permanecerá inalterada.

In [0]:

ufs_validas = [
    "AC", "AL", "AP", "AM", "BA", "CE", "DF",
    "ES", "GO", "MA", "MT", "MS", "MG", "PA",
    "PB", "PR", "PE", "PI", "RJ", "RN", "RS",
    "RO", "RR", "SC", "SP", "SE", "TO"
]

df_validado = (
    df_sem_duplicatas
    .withColumn(
        "erro_identificador",
        F.col("cod_perturbacao").isNull() |
        (F.col("cod_perturbacao") == "")
    )
    .withColumn(
        "erro_data",
        F.col("din_interrupcaocarga").isNull()
    )
    .withColumn(
        "erro_subsistema",
        F.col("id_subsistema").isNull() |
        (F.col("id_subsistema") == "")
    )
    .withColumn(
        "erro_uf",
        F.col("id_estado").isNull() |
        (~F.col("id_estado").isin(ufs_validas))
    )
    .withColumn(
        "erro_carga",
        F.col("val_cargainterrompida_mw").isNull() |
        (F.col("val_cargainterrompida_mw") < 0)
    )
    .withColumn(
        "erro_tempo",
        F.col("val_tempomedio_minutos").isNull() |
        (F.col("val_tempomedio_minutos") < 0)
    )
    .withColumn(
        "erro_energia",
        F.col("val_energianaosuprida_mwh").isNull() |
        (F.col("val_energianaosuprida_mwh") < 0)
    )
    .withColumn(
        "erro_rede_basica",
        F.col("flg_envolveuredebasica").isNull() |
        (~F.col("flg_envolveuredebasica").isin("S", "N"))
    )
    .withColumn(
        "erro_rede_operacao",
        F.col("flg_envolveuredeoperacao").isNull() |
        (~F.col("flg_envolveuredeoperacao").isin("S", "N"))
    )
)


In [0]:

colunas_erro = [
    "erro_identificador",
    "erro_data",
    "erro_subsistema",
    "erro_uf",
    "erro_carga",
    "erro_tempo",
    "erro_energia",
    "erro_rede_basica",
    "erro_rede_operacao"
]

df_validado = df_validado.withColumn(
    "registro_valido",
    ~F.expr(
        " OR ".join(colunas_erro)
    )
)

df_validado = df_validado.withColumn(
    "motivo_rejeicao",
    F.concat_ws(
        "; ",
        *[
            F.when(F.col(c), F.lit(c))
            for c in colunas_erro
        ]
    )
)

display(
    df_validado.groupBy("registro_valido").count()
)


In [0]:

df_invalidos = df_validado.filter(
    F.col("registro_valido") == False
)

df_silver = df_validado.filter(
    F.col("registro_valido") == True
)

print("Registros válidos:", df_silver.count())
print("Registros inválidos:", df_invalidos.count())

display(
    df_invalidos.select(
        "cod_perturbacao",
        "motivo_rejeicao",
        "val_cargainterrompida_mw",
        "val_tempomedio_minutos",
        "val_energianaosuprida_mwh"
    )
)


## Enriquecimento temporal

A data e hora da interrupção será utilizada para criar atributos auxiliares de análise.

Serão extraídos ano, mês, trimestre, dia e hora, permitindo análises históricas e sazonais.

Os atributos derivados não substituem a data original. Eles representam informações adicionais para facilitar consultas e a posterior construção da dimensão tempo.

In [0]:

df_silver = (
    df_silver
    .withColumn(
        "data_interrupcao",
        F.to_date("din_interrupcaocarga")
    )
    .withColumn(
        "ano_interrupcao",
        F.year("din_interrupcaocarga")
    )
    .withColumn(
        "mes_interrupcao",
        F.month("din_interrupcaocarga")
    )
    .withColumn(
        "trimestre_interrupcao",
        F.quarter("din_interrupcaocarga")
    )
    .withColumn(
        "dia_interrupcao",
        F.dayofmonth("din_interrupcaocarga")
    )
    .withColumn(
        "hora_interrupcao",
        F.hour("din_interrupcaocarga")
    )
)

display(df_silver.limit(10))


In [0]:

df_silver_final = df_silver.drop(
    *colunas_erro,
    "registro_valido",
    "motivo_rejeicao"
)

(
    df_silver_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_SILVER)
)

(
    df_invalidos.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_INVALIDOS)
)

print("Tabelas Silver persistidas.")


In [0]:

spark.sql(f"""
COMMENT ON TABLE {TABELA_SILVER}
IS 'Registros tratados de interrupcao de carga do ONS, com padronizacao, deduplicacao, validacao e atributos temporais.'
""")

spark.sql(f"""
COMMENT ON TABLE {TABELA_INVALIDOS}
IS 'Registros rejeitados pelas regras de qualidade, preservados para auditoria e rastreabilidade.'
""")


In [0]:

qtd_bronze = spark.table(TABELA_ORIGEM).count()

qtd_silver = spark.table(TABELA_SILVER).count()

qtd_invalidos = spark.table(TABELA_INVALIDOS).count()

qtd_duplicatas = qtd_antes - qtd_depois

total_reconciliado = (
    qtd_silver +
    qtd_invalidos +
    qtd_duplicatas
)

print("Bronze:", qtd_bronze)
print("Silver:", qtd_silver)
print("Inválidos:", qtd_invalidos)
print("Duplicatas:", qtd_duplicatas)
print("Total reconciliado:", total_reconciliado)

assert qtd_bronze == total_reconciliado

print("Reconciliação concluída com sucesso!")
